# ForgeLM — Post-Training Workflow (Phase 10)

Once you've fine-tuned a model, ForgeLM's Phase 10 toolchain turns it
into something deployable. This notebook walks through the four CLI
commands that run **after** training:

1. **`forgelm --fit-check`** — estimate VRAM before training so you don't
   discover OOM 30 minutes in.
2. **`forgelm chat`** — interactive REPL to sanity-check the trained model.
3. **`forgelm export`** — convert HuggingFace weights to GGUF for
   Ollama / llama.cpp / compatible runtimes.
4. **`forgelm deploy`** — generate a ready-to-use config for vLLM, TGI,
   Ollama, or HuggingFace Inference Endpoints.

Programmatic access is also available via `forgelm.inference`.

**Prerequisites:** Either a model fine-tuned with ForgeLM (see
`quickstart_sft.ipynb`) under `./checkpoints/final_model`, or any
HuggingFace model directory.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/post_training_workflow.ipynb)

In [ ]:
# Install ForgeLM (skip on Colab if quickstart_sft already ran in this session)
!pip install -q --no-cache-dir 'forgelm==0.5.5' bitsandbytes
!forgelm --version

## 1. VRAM Fit-Check (run **before** training)

`--fit-check` estimates peak training VRAM from your config and the
model architecture, *without* loading any weights. It returns one of
four verdicts:

| Verdict | Meaning |
|---------|---------|
| `FITS` | Estimated total ≤ 85% of available VRAM — should run |
| `TIGHT` | 85–95% — likely to run but no headroom for optimizer spikes |
| `OOM` | > 95% — will fail; recommendations printed |
| `UNKNOWN` | No GPU detected (hypothetical mode) |

It also prints a per-component breakdown (base weights / LoRA adapter /
optimizer state / activations / GaLore buffers) and an `estimation_caveat`
noting that FlashAttention/SDPA, MoE expert counts, and sliding-window
attention aren't modeled — treat the verdict as advisory.

In [ ]:
# Plain-text output (default)
!forgelm --config config_template.yaml --fit-check

# JSON output for CI/CD or scripting
!forgelm --config config_template.yaml --fit-check --output-format json

## 2. Interactive Chat REPL

`forgelm chat MODEL_PATH` loads a fine-tuned model and starts a
terminal session. Slash commands:

| Command | Effect |
|---------|--------|
| `/reset` | Clear conversation history |
| `/save [path]` | Save transcript as JSONL (system prompt included) |
| `/temperature N` | Change sampling temperature mid-session |
| `/system PROMPT` | Set or update the system prompt |
| `/exit`, `/quit` | End session |

Optional flags: `--adapter PATH` to merge a LoRA adapter, `--load-in-4bit`,
`--no-stream` to disable token streaming, `--backend unsloth`.

> **Notebook note:** chat is interactive, so it won't run inside a
> notebook cell. The example below is the command you'd run in a
> terminal once training finishes.

In [ ]:
# Terminal example — do not run inside the notebook
# $ forgelm chat ./checkpoints/final_model --temperature 0.7
# $ forgelm chat ./checkpoints/final_model --adapter ./adapters/my-lora --load-in-4bit
print("forgelm chat is interactive; run it from a terminal.")

### Programmatic generation (`forgelm.inference`)

If you need generation inside Python (e.g. for batch eval or a custom
API), the same primitives the chat REPL uses are exposed as a small
module:

In [ ]:
from forgelm.inference import generate, generate_stream, load_model

MODEL_PATH = "./checkpoints/final_model"  # or any HF Hub ID

# `load_model` returns a (model, tokenizer) pair. Adapter merge happens
# automatically when `adapter=...` is given; pass `backend="unsloth"` for
# the Unsloth fast path.
model, tokenizer = load_model(
    MODEL_PATH,
    adapter=None,  # path to a PEFT adapter, or None
    backend="transformers",  # or "unsloth"
    load_in_4bit=False,  # QLoRA-style 4-bit NF4 inference
)

# Single-shot generation
answer = generate(
    model,
    tokenizer,
    prompt="Explain LoRA in one sentence.",
    system_prompt="You are a concise machine-learning tutor.",
    max_new_tokens=120,
    temperature=0.7,
)
print(answer)

# Streaming — yields decoded text fragments as they're produced
for chunk in generate_stream(
    model,
    tokenizer,
    prompt="List three pitfalls of fine-tuning small LMs.",
    max_new_tokens=200,
):
    print(chunk, end="", flush=True)
print()

## 3. GGUF Export

`forgelm export` wraps `llama-cpp-python`'s bundled `convert_hf_to_gguf.py`
to produce GGUF files for Ollama / llama.cpp / compatible runtimes.

**K-quant note:** the script supports `f16` and `q8_0` directly. K-quants
(`q2_k` / `q3_k_m` / `q4_k_m` / `q5_k_m`) need a **two-step** flow — ForgeLM
produces an intermediate `.f16.gguf` first; you then run `llama-quantize`
to get the K-quant. The CLI prints the exact follow-up command. The
`ExportResult.quant` field always reflects what was actually written
(so `model_integrity.json` SHA-256 stays honest).

Install the optional extra first:

In [ ]:
!pip install -q 'forgelm[export]==0.5.5'  # adds llama-cpp-python>=0.2.90

In [ ]:
# Direct support: f16 / q8_0
!forgelm export ./checkpoints/final_model --output ./model.q8.gguf --quant q8_0

# K-quant (two-step) — the warning will tell you the llama-quantize command
!forgelm export ./checkpoints/final_model --output ./model.q4_k_m.gguf --quant q4_k_m
# After the export above finishes, run:
#   llama-quantize ./model.q4_k_m.f16.gguf ./model.q4_k_m.gguf Q4_K_M

# If you trained with LoRA, --adapter merges before conversion
# !forgelm export ./base_model --adapter ./checkpoints/final_model --output ./merged.gguf --quant f16

## 4. Deployment Configs

`forgelm deploy --target T --output FILE` writes a ready-to-use config
for one of four serving runtimes — it does **not** start the server.

| Target | Output | Mounts model from |
|--------|--------|-------------------|
| `ollama` | Modelfile (`ollama create -f Modelfile`) | local dir |
| `vllm` | `vllm_config.yaml` | local dir or HF Hub ID |
| `tgi` | `docker-compose.yaml` | local dir |
| `hf-endpoints` | API spec JSON | HF Hub repo ID |

`tgi` and `ollama` reject HF Hub IDs up front (those would silently produce
broken volumes). `vllm` and `hf-endpoints` accept either.

In [ ]:
# Ollama Modelfile (system prompt embedded, sampling defaults baked in)
!forgelm deploy ./checkpoints/final_model --target ollama --output ./Modelfile \
    --system "You are a helpful coding assistant."
!head -20 Modelfile

# vLLM engine config (YAML)
!forgelm deploy ./checkpoints/final_model --target vllm --output ./vllm.yaml \
    --max-length 4096 --gpu-memory-utilization 0.85

# HuggingFace Inference Endpoints (JSON spec; vendor configurable)
!forgelm deploy your-org/your-model --target hf-endpoints --output ./endpoint.json \
    --vendor aws  # or azure, gcp

## Compliance Artifacts (Phase 8 + Phase 10 integration)

Every export updates `model_integrity.json` with the new artifact's
path, format, quant, SHA-256, and size in bytes. This file is part of the
audit trail required by EU AI Act Article 15 — deployers can verify the
GGUF they pulled matches the one ForgeLM signed off on.

Use `--no-integrity-update` to skip the manifest update for one-off
experimental exports.

## Where next

- Data preparation: [`data_curation.ipynb`](./data_curation.ipynb) — `forgelm ingest` + `forgelm audit` end-to-end (raw docs → SFT-ready JSONL with PII / secrets masking + audit JSON for compliance).
- Training notebooks: `quickstart_sft.ipynb`, `dpo_alignment.ipynb`,
  `grpo_reasoning.ipynb`, `kto_binary_feedback.ipynb`,
  `galore_memory_optimization.ipynb`, `multi_dataset.ipynb`,
  `synthetic_data_training.ipynb`
- Eval notebook: `safety_evaluation.ipynb`
- Full configuration reference: [docs/reference/configuration.md](https://github.com/cemililik/ForgeLM/blob/main/docs/reference/configuration.md)
- Release history: [CHANGELOG.md](https://github.com/cemililik/ForgeLM/blob/main/CHANGELOG.md)